# Лабораторный практикум по курсу «Синтез Речи»
### Лабораторная работа №1. Нормализация текста

**Выполнила:** Ходжаметова Милана Илимдаровна

**Цель работы:** определение ненормализованных текстовых данных в корпусе RUSLAN и разработка средств их обработки — фильтра нормализованности и нормализатора текста.

**Краткое описание:** в работе проводится разведочный анализ корпуса RUSLAN и отладочного набора dev_sentences.csv, реализуется rule-based классификатор TextFilter, различающий нормализованные и ненормализованные предложения, разрабатывается TextNormalizer, выполняющий безопасные character-level правки, и собирается пайплайн preprocess_ruslan.py, формирующий очищенный файл metadata_RUSLAN_22200_normalized.csv.

## Этап 1. Скачивание и подготовка корпуса

In [3]:
import csv
import pandas as pd

df_corpus = pd.read_csv(
    "../../data/RUSLAN/metadata_RUSLAN_22200.csv",
    sep="|", header=None, names=["audio_id", "text"],
    dtype=str, encoding="utf-8",
    quoting=csv.QUOTE_NONE, keep_default_na=False,
)
print(df_corpus.shape)
df_corpus.head()

(22200, 2)


,audio_id,text
0,000000_RUSLAN,С тревожным чувством берусь я за перо.
1,000001_RUSLAN,Кого интересуют признания литературного неудач...
2,000002_RUSLAN,Что поучительного в его исповеди?
3,000003_RUSLAN,Да и жизнь моя лишена внешнего трагизма.
4,000004_RUSLAN,Я абсолютно здоров.


## Этап 2. Разведочный анализ (EDA)

### 2.1. Базовая статистика

In [4]:
df_corpus["n_chars"] = df_corpus["text"].str.len()
df_corpus["n_words"] = df_corpus["text"].str.split().str.len()

print("Длина в символах:")
print(df_corpus["n_chars"].describe().round(2))
print("\nДлина в словах:")
print(df_corpus["n_words"].describe().round(2))
print(f"\nПустых строк: {(df_corpus['text'].str.strip() == '').sum()}")

Длина в символах:
count    22200.00
mean        80.79
std         41.50
min         10.00
25%         55.00
50%         85.00
75%        102.00
max        723.00
Name: n_chars, dtype: float64

Длина в словах:
count    22200.00
mean        12.03
std          6.23
min          1.00
25%          8.00
50%         12.00
75%         15.00
max        111.00
Name: n_words, dtype: float64

Пустых строк: 0


### 2.2. Символьный анализ

Всего в корпусе **91 уникальный символ**. Из них нестандартных — 12.

In [5]:
import unicodedata
from collections import Counter

all_chars = Counter()
for t in df_corpus["text"]:
    all_chars.update(t)

print(f"Всего уникальных символов: {len(all_chars)}")
print("\nТоп-30 по частоте:")
for ch, cnt in all_chars.most_common(30):
    name = unicodedata.name(ch, "?")
    print(f"  {repr(ch):8} {cnt:8}  {name}")

Всего уникальных символов: 91

Топ-30 по частоте:
  ' '        260368  SPACE
  'о'        147022  CYRILLIC SMALL LETTER O
  'е'        121873  CYRILLIC SMALL LETTER IE
  'а'        119548  CYRILLIC SMALL LETTER A
  'и'        100848  CYRILLIC SMALL LETTER I
  'н'         88160  CYRILLIC SMALL LETTER EN
  'т'         85588  CYRILLIC SMALL LETTER TE
  'с'         71415  CYRILLIC SMALL LETTER ES
  'л'         69883  CYRILLIC SMALL LETTER EL
  'р'         67289  CYRILLIC SMALL LETTER ER
  'в'         57791  CYRILLIC SMALL LETTER VE
  'к'         47225  CYRILLIC SMALL LETTER KA
  'м'         43795  CYRILLIC SMALL LETTER EM
  'д'         40574  CYRILLIC SMALL LETTER DE
  'у'         40359  CYRILLIC SMALL LETTER U
  '.'         32987  FULL STOP
  'п'         31772  CYRILLIC SMALL LETTER PE
  'я'         28582  CYRILLIC SMALL LETTER YA
  'ы'         28014  CYRILLIC SMALL LETTER YERU
  'ь'         25399  CYRILLIC SMALL LETTER SOFT SIGN
  'з'         25343  CYRILLIC SMALL LETTER ZE
  'г'        

In [6]:
STANDARD_PUNCT = set(".,!?:;\"'«»…—- ")

def get_non_standard_chars(text):
    result = []
    for ch in text:
        if "CYRILLIC" in unicodedata.name(ch, "") or ch == "\u0301":
            continue
        if ch in STANDARD_PUNCT:
            continue
        if ch.isdigit() or "LATIN" in unicodedata.name(ch, ""):
            continue
        result.append(ch)
    return result

df_corpus["non_standard_chars"] = df_corpus["text"].apply(get_non_standard_chars)
non_std_counter = Counter(ch for lst in df_corpus["non_standard_chars"] for ch in lst)

n_lines_with_anomaly = (df_corpus["non_standard_chars"].str.len() > 0).sum()
print(f"Строк с нестандартными символами: {n_lines_with_anomaly} ({n_lines_with_anomaly/len(df_corpus)*100:.2f}%)")
print(f"Уникальных нестандартных символов: {len(non_std_counter)}\n")
for ch, cnt in non_std_counter.most_common():
    print(f"  {repr(ch):8} {cnt:8}  {unicodedata.name(ch, '?')}")

Строк с нестандартными символами: 6999 (31.53%)
Уникальных нестандартных символов: 12

  '–'         10410  EN DASH
  '‑'          2243  NON-BREAKING HYPHEN
  '('           305  LEFT PARENTHESIS
  ')'           305  RIGHT PARENTHESIS
  '„'            72  DOUBLE LOW-9 QUOTATION MARK
  '“'            57  LEFT DOUBLE QUOTATION MARK
  '”'            13  RIGHT DOUBLE QUOTATION MARK
  '’'             6  RIGHT SINGLE QUOTATION MARK
  '/'             4  SOLIDUS
  '*'             1  ASTERISK
  '<'             1  LESS-THAN SIGN
  '>'             1  GREATER-THAN SIGN


### 2.3. Примеры аномалий

**EN DASH (`–`)** — тире в середине предложения и начало реплики диалога.

**NON-BREAKING HYPHEN (`‑`)** — неразрывный дефис в наречиях.

**Немецкие кавычки (`„ “ ” ’`)** — типографские кавычки.

**Круглые скобки** — авторские вставки.

**Технический мусор** (`* / < >`) — артефакты вёрстки.

In [8]:
def show_examples(pattern, title, n=5):
    print(f"=== {title} ===")
    mask = df_corpus["text"].str.contains(pattern, regex=True, na=False)
    for t in df_corpus[mask]["text"].head(n):
        print(f" • {t}")
    print()

show_examples("–", "EN DASH")
show_examples("‑", "NON-BREAKING HYPHEN")
show_examples(r"[„“”’]", "Немецкие кавычки")
show_examples(r"[()]", "Круглые скобки")
show_examples(r"[*/<>]", "Технический мусор")

=== EN DASH ===
 • Ходить мимо отгороженных проволочными сетками вольеров – было небезопасно. 
 • А пьянка – это неминуемое чепе… Если бы ты постарался, употребил, как говорится, свое влияние… Поговори с Балодисом, Воликовым… Ну и, конечно, с Петровым.
 • – Приближается Новый год. К сожалению, это неизбежно. Значит, в казарме будет пьянка. 
 • Главный тезис – пей, но знай меру. Вообще не пить – это слишком. Это, как говорится, антимарксистская утопия. 
 • В тот же день Борис заметил около уборной ефрейтора Петрова, которого сослуживцы называли – Фидель. 

=== NON-BREAKING HYPHEN ===
 • Моя жена говорит, если бы ты оделся как следует… Между прочим, его жена звонит как‑то раз… Стоп! 
 • С каким‑то идиотским простодушием он добавил Человек двадцать претендовало на место. 
 • И все же какие‑то люди стоят за этим, какие‑то разговоры, чувства, действительность… Не в самих листах, а там, на горизонте…
 • Кое‑что перечитал. Задумался… Пожелтевшие листы. Десять лет вранья и притворства. 
 • За 

### 2.4. Выводы EDA

Корпус RUSLAN представляет собой **литературную прозу**, а не диалоговые фразы. Классические типы ненормализованности (цифры, латиница, формулы, эмодзи) в нём практически отсутствуют: цифры — в 4 строках из 22 200, латиницы нет вовсе.

Основная масса аномалий — **типографические**:

| Категория | Кол-во | Решение |
|---|---:|---|
| EN DASH `–` | 10 410 | **не трогать** — норма русского языка |
| NON-BREAKING HYPHEN `‑` | 2 243 | нормализовать в `-` |
| Круглые скобки `( )` | 305 пар | **не трогать** — авторские вставки |
| Немецкие кавычки `„“”’` | 148 | нормализовать в `«»` / `'` |
| Технический мусор | 7 | отбрасывать фильтром |

**Итог:** основная работа ложится на нормализатор; фильтр нужен только для редких настоящих аномалий. Нельзя удалять `–` и круглые скобки — это норма.

## Этап 3. Фильтр `TextFilter`

### 3.1. Концепция

`TextFilter` — rule-based классификатор на регулярных выражениях. Возвращает `1` для нормализованного текста и `0` для ненормализованного. Использует три группы признаков:

1. **Символьные**: цифры, латиница, техсимволы, невидимые символы.
2. **Пунктуационные**: пробел перед знаком, знак без пробела после, повтор `??`/`!!`, `!.`, `..` (но не `...`), несбалансированные скобки.
3. **Семантические**: сокращения (`ул.`, `г-н`, `г-жа`), инициалы (`С. Кузнецов`), короткие аббревиатуры (`МГУ`, `КПСС`), междометия (`ммм`, `хм`, `хехе`), смайлики, смесь алфавитов в слове (`Tариф`).

Дополнительно есть проверка через `unicodedata.name()`: если символ вне (кириллица + знак ударения + стандартная пунктуация), строка отбрасывается.

### 3.2. Итерации разработки

| Итерация | F1 на dev | Изменение |
|---|---:|---|
| Baseline (12 правил) | 0.9527 | старт |
| + `IGNORECASE` для междометий и `г-н/г-жа` | 0.9792 | учли регистр |
| + падежи `г-жи`, словарь `Минюст, Минздрав…` | 0.9826 | +0.003 |
| + белый список `unicodedata` | 0.9826 | надежность |
| Ослабление кавычек, скобок, аббревиатур под RUSLAN | **0.9827** | не режем нормальную прозу |

In [9]:
from text_filter import TextFilter
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report

dev = pd.read_csv("data/dev_sentences.csv", sep="|", encoding="utf-8",
                  quoting=csv.QUOTE_NONE)

tf = TextFilter()
dev["predicted"] = dev["text"].apply(tf.filter)

f1 = f1_score(dev["is_normalized"], dev["predicted"])
prc = precision_score(dev["is_normalized"], dev["predicted"])
rec = recall_score(dev["is_normalized"], dev["predicted"])

print(f"F1:        {f1:.4f}")
print(f"Precision: {prc:.4f}")
print(f"Recall:    {rec:.4f}\n")
print(classification_report(dev["is_normalized"], dev["predicted"], digits=4))

F1:        0.9827
Precision: 0.9793
Recall:    0.9861

              precision    recall  f1-score   support

           0     0.9871    0.9808    0.9839       156
           1     0.9793    0.9861    0.9827       144

    accuracy                         0.9833       300
   macro avg     0.9832    0.9834    0.9833       300
weighted avg     0.9834    0.9833    0.9833       300



### 3.3. Результаты на dev-наборе

| Метрика | Значение | Порог |
|---|---:|---:|
| F1 | **0.9827** | ≥ 0.9 |
| Precision | 0.9793 | — |
| Recall | 0.9861 | — |

Баланс классов: 156 ненормализованных против 144 нормализованных. Ошибок всего 5 — по большей части это шум в разметке (строки с `?!.` помечены как нормализованные).

## Этап 4. Нормализатор `TextNormalizer`

### 4.1. Концепция

Нормализатор выполняет **только безопасные character-level правки** — те, что не меняют произношение слов и не нарушают соответствия текста аудио. Список правил в порядке применения:

1. Unicode NFC (не удаляет `U+0301`)
2. Невидимые символы (`\u200b`, `\u200c`, `\u200d`, `\ufeff`, `\u00a0`, `\u202f`) → пробел
3. Неразрывный дефис `‑` (U+2011), мягкий перенос (U+00AD) → обычный `-`
4. Типографские кавычки `„` → `«`, `“`/`”` → `»`, `’`/`` ` `` → `'`
5. Сломанная пунктуация: `!..` → `!`, `?..` → `?`, `..` → `.`, `....+` → `…`
6. Пробел перед знаком удаляется, множественные пробелы схлопываются

### 4.2. Что нормализатор НЕ делает (сознательно)

- Не трогает `–` (EN DASH) — норма русского языка
- Не трогает круглые скобки — авторские вставки
- Не трогает `ё` и знак ударения `U+0301`
- Не раскрывает сокращения — нарушит соответствие аудио
- Не удаляет техмусор (`* / < >`) — такие строки отбрасывает фильтр

In [10]:
from text_normalizer import TextNormalizer

nrm = TextNormalizer()
samples = [
    "Расстреливать надо таких писателей!.",
    "Я на букву О /библиография к Окуджаве/.",
    "де\u2011факто",
    "„повестью в диалогах”",
    "Текст , ещё текст.",
    "Многоточие... и точки..",
    "Абзац с\u00a0неразрывным пробелом.",
    "Слово\u200bс невидимым символом.",
    "Стресс: за́мок и замо́к.",
]
for s in samples:
    print(f"IN : {s!r}")
    print(f"OUT: {nrm.normalize(s)!r}")
    print()

IN : 'Расстреливать надо таких писателей!.'
OUT: 'Расстреливать надо таких писателей!'

IN : 'Я на букву О /библиография к Окуджаве/.'
OUT: 'Я на букву О /библиография к Окуджаве/.'

IN : 'де‑факто'
OUT: 'де-факто'

IN : '„повестью в диалогах”'
OUT: '«повестью в диалогах»'

IN : 'Текст , ещё текст.'
OUT: 'Текст, ещё текст.'

IN : 'Многоточие... и точки..'
OUT: 'Многоточие... и точки.'

IN : 'Абзац с\xa0неразрывным пробелом.'
OUT: 'Абзац с неразрывным пробелом.'

IN : 'Слово\u200bс невидимым символом.'
OUT: 'Слово с невидимым символом.'

IN : 'Стресс: за́мок и замо́к.'
OUT: 'Стресс: за́мок и замо́к.'



## Этап 5. Очистка корпуса и сохранение результата

### 5.1. Пайплайн

1. Загрузить `data/RUSLAN/metadata_RUSLAN_22200.csv`
2. Применить `TextNormalizer` к каждой строке
3. Применить `TextFilter` к нормализованному тексту
4. Сохранить отфильтрованные строки в `data/metadata_RUSLAN_22200_normalized.csv` в формате `id|raw|normalized`

In [11]:
df = df_corpus.copy()

df["nrm"] = df["text"].apply(nrm.normalize)
df["kept"] = df["nrm"].apply(tf.filter)

total = len(df)
kept = int(df["kept"].sum())
dropped = total - kept
changed = int(((df["kept"] == 1) & (df["text"] != df["nrm"])).sum())

print(f"Всего записей:     {total}")
print(f"Оставлено:         {kept} ({kept/total*100:.2f}%)")
print(f"Отброшено:         {dropped} ({dropped/total*100:.2f}%)")
print(f"Изменено нормализатором среди оставленных: {changed}")

Всего записей:     22200
Оставлено:         16133 (72.67%)
Отброшено:         6067 (27.33%)
Изменено нормализатором среди оставленных: 10185


### 5.2. Результаты

| Метрика | Значение |
|---|---:|
| Всего на входе | 22 200 |
| Оставлено после очистки | **16 133 (72,67 %)** |
| Отброшено | 6 067 (27,33 %) |

Порог приёмки **≥ 16 000** выполнен: 16 133 ≥ 16 000.

Основные причины отбраковки: одиночные `»`/`«` без парной кавычки (цитаты разорваны между фразами), повторы `??`/`!!`/`?!`, капс-заголовки длиной более 4 букв, техсимволы, инициалы.

In [12]:
import unicodedata

path = "../../data/metadata_RUSLAN_22200_normalized.csv"
with open(path, encoding="utf-8") as f:
    bad_nfc = sum(1 for line in f if unicodedata.normalize("NFC", line) != line)
with open(path, encoding="utf-8") as f:
    bad_cols = sum(1 for line in f if line.count("|") != 2)

print(f"Не в NFC:         {bad_nfc}")
print(f"Не с 3 колонками: {bad_cols}")

Не в NFC:         0
Не с 3 колонками: 0


In [13]:
changed_mask = (df["kept"] == 1) & (df["text"] != df["nrm"])
for _, row in df[changed_mask][["text", "nrm"]].head(10).iterrows():
    print(f"RAW: {row['text']}")
    print(f"NRM: {row['nrm']}\n")

RAW: Вы страшное говно, мон колонель, не обессудьте!..
NRM: Вы страшное говно, мон колонель, не обессудьте!

RAW: Какать в одном поле не сяду!..
NRM: Какать в одном поле не сяду!

RAW: Да это же писатель!..
NRM: Да это же писатель!

RAW: Расстреливать надо таких писателей!..
NRM: Расстреливать надо таких писателей!

RAW: Расстреливать надо таких писателей!.. Кажется, я допускаю ошибку.
NRM: Расстреливать надо таких писателей! Кажется, я допускаю ошибку.

RAW: Мало ли загадочных типов шатается по бульварам?..
NRM: Мало ли загадочных типов шатается по бульварам?

RAW:  А ты славный малый!
NRM: А ты славный малый!

RAW: И я пишу, обрадовался брат Махнем не глядя?..
NRM: И я пишу, обрадовался брат Махнем не глядя?

RAW: Ничего страшного, я потом вымою руки..
NRM: Ничего страшного, я потом вымою руки.

RAW: Вы ошибаетесь! 
NRM: Вы ошибаетесь!



## Итоговые выводы

### Соответствие критериям

| Критерий | Требование | Результат |
|---|---|---|
| F1 на открытом множестве | ≥ 0.9 | **0.9827** |
| F1 на закрытом множестве | ≥ 0.85 | ожидается |
| Сохранено фраз | ≥ 16 000 | **16 133** |
| Формат файла | 3 колонки, UTF-8, NFC |  |

### Что сделано

1. Проведён разведочный анализ корпуса RUSLAN и dev-набора. Показано, что корпус — литературная проза с преобладанием типографических аномалий (`–`, `‑`, `„“”`).
2. Разработан `TextFilter` с F1 = 0.9827 на dev-наборе.
3. Разработан `TextNormalizer` с безопасными character-level правками.
4. Собран пайплайн `preprocess_ruslan.py`, сохранивший 16 133 нормализованные фразы.
5. Проверено соответствие формата выходного файла требованиям приёмки.

### Артефакты

- `labs/lab1_text/text_filter.py`
- `labs/lab1_text/text_normalizer.py`
- `labs/lab1_text/preprocess_ruslan.py`
- `labs/lab1_text/report.ipynb` 
- `data/metadata_RUSLAN_22200_normalized.csv` (16 133 строки)